In [5]:
# ---------------------------------------------------------
# Rahul Aloth Rajan
# Training for the Disparity Calculation from Left and Right Camera Images.
# ---------------------------------------------------------
# Mini-RAFT Stereo Training Script
# ---------------------------------------------------------
# This script trains (or fine-tunes) the Mini-RAFT Stereo model
# on the KITTI or SceneFlow dataset. It handles:
#   - dataset loading and preprocessing
#   - stereo augmentations and transforms
#   - forward/backward passes
#   - loss computation and optimization
#   - checkpoint saving and logging
#
# Usage:
# This is a ipython notebook where Libraries are implimented as a backend layer for training and validation.
#
# This file is part of the Stereo Depth Estimation project.
# ---------------------------------------------------------

# ---------------------------------------------------------

In [6]:
# ---------------------------------------------------------
# 1. Imports
# ---------------------------------------------------------
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as T

from data.kitti_dataset import KITTIStereo
from transforms.stereo_augment import StereoAugmentation
from transforms.stereo_compose import StereoCompose
from models.mini_raft_stereo import MiniRAFTStereo


In [8]:
# 2. Create transforms
# Image transforms (ColorJitter + ToTensor)
# This creates a pipeline of appearance-based augmentations that will be
# applied to each image (left and right) after geometric stereo augmentations.
# Image goes through ColorJitter → then ToTensor.
# ColorJitter - This randomly changes the appearance of the image:
# T.ToTensor() - This converts the PIL image into a PyTorch tensor. pixel values go from 0–255 → 0–1
image_transforms = T.Compose([
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.5),
    T.ToTensor()
])

# Full stereo transform pipeline
# StereoAugmentation -> This applies geometric transforms that must be applied identically to left, right , disparity image
# Ensures all samples have the same size for training.(288, 576).
# Simulates mirrored scenes (left/right swapped).
# StereoCompose -> This applies appearance transforms to the images and converts to Tensor( disparity -> Tensor, mask -> Tensor)
train_transform = T.Compose([
    StereoAugmentation(output_size=(288, 576), do_flip=True),
    StereoCompose(image_transforms)
])


In [9]:
# 3. Create dataset + dataloader
## The link to download the sample : https://www.cvlibs.net/datasets/kitti/eval_scene_flow.php?benchmark=stereo
# Data validation code snippet.
import os
import glob

dataset_root = "/home/rahul/AiStätt/RAFT/data/data_scene_flow/"   # KIITTI path local.
left_paths = sorted(glob.glob(os.path.join(dataset_root, "training/image_2/*.png")))
right_paths = sorted(glob.glob(os.path.join(dataset_root, "training/image_3/*.png")))
disp_paths = sorted(glob.glob(os.path.join(dataset_root, "training/disp_occ_0/*.png")))

print (len(left_paths))
print (len(right_paths))
print (len(disp_paths))

assert len(left_paths) == len(right_paths), "Left/right mismatch"
assert len(left_paths) == len(disp_paths), "Image/disparity mismatch"


200
200
200


In [10]:
dataset_root = "/home/rahul/AiStätt/RAFT/data/data_scene_flow/"   # KIITTI path local.
# This creates a PyTorch dataset object for the KITTI stereo training data,
# and it attaches your full augmentation + preprocessing pipeline to it.
train_dataset = KITTIStereo(root_dir=dataset_root, transform=train_transform)

# PyTorch DataLoader which feeds stereo samples into which model during training.
train_loader = DataLoader(
     train_dataset,
     batch_size=1,
     shuffle=True,
     num_workers=2,
     drop_last=True
 )

print("Samples:", len(train_dataset))


Samples: 200


In [25]:
# 4. Create Mini‑RAFT model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniRAFTStereo(
    iters=8,
    max_disp=4,
    num_corr_levels=3,
    feature_base_ch=32
).to(device)

print("GRU convz weight shape:", model.update_block.gru.convz.weight.shape)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)

print("Model parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")


GRU convz weight shape: torch.Size([96, 188, 3, 3])
Model parameters: 0.988833 M


In [26]:
#5. RAFT multi‑step loss
def raft_multistep_loss(preds, gt_disp, valid_mask):
    """
    preds: list of disparity predictions [B,1,H/4,W/4]
    gt_disp: ground truth disparity (downsampled)
    valid_mask: 0/1 mask (downsampled)
    """
    loss = 0.0
    n = len(preds)

    for i, pred in enumerate(preds):
        weight = 0.8 ** (n - i - 1)  # later predictions get higher weight
        diff = (pred - gt_disp).abs() * valid_mask
        denom = valid_mask.sum() + 1e-6
        loss_i = diff.sum() / denom
        loss += weight * loss_i

    return loss


In [27]:
# 6. Training loop
num_epochs = 10
grad_clip = 1.0

model.train()

for epoch in range(num_epochs):
    for step, batch in enumerate(train_loader):

        left = batch["left"].to(device)          # [B,3,H,W]
        right = batch["right"].to(device)        # [B,3,H,W]
        disp_gt = batch["disparity"].to(device)  # [B,1,H,W]
        valid = batch["valid_mask"].to(device)   # [B,1,H,W]

        # -----------------------------
        # Debug: verify shapes once
        # -----------------------------
        if step == 0:
            print("LEFT:", left.shape)
            print("RIGHT:", right.shape)
            print("DISP_GT:", disp_gt.shape)
            print("VALID:", valid.shape)

        # -----------------------------
        # Downsample GT to 1/4 resolution
        # -----------------------------
        disp_gt_ds = F.interpolate(disp_gt, scale_factor=0.25, mode="nearest")
        valid_ds = F.interpolate(valid, scale_factor=0.25, mode="nearest")

        # -----------------------------
        # Forward pass
        # -----------------------------
        preds = model(left, right)   # list of [B,1,H/4,W/4]

        # Debug: print shape of last prediction
        if step == 0:
            print("PRED SHAPE:", preds[-1].shape)

        # -----------------------------
        # Loss
        # -----------------------------
        loss = raft_multistep_loss(preds, disp_gt_ds, valid_ds)

        # -----------------------------
        # Backprop
        # -----------------------------
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        # -----------------------------
        # Logging
        # -----------------------------
        if step % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Step {step} | Loss: {loss.item():.4f}")
        # saving the mode
        torch.save(model.state_dict(), f"mini_raft_epoch_{epoch+1}.pth")

LEFT: torch.Size([1, 3, 288, 576])
RIGHT: torch.Size([1, 3, 288, 576])
DISP_GT: torch.Size([1, 1, 288, 576])
VALID: torch.Size([1, 1, 288, 576])
PRED SHAPE: torch.Size([1, 1, 72, 144])
Epoch 1/10 | Step 0 | Loss: 102.5143
Epoch 1/10 | Step 10 | Loss: 51.1618
Epoch 1/10 | Step 20 | Loss: 64.5492
Epoch 1/10 | Step 30 | Loss: 75.6563
Epoch 1/10 | Step 40 | Loss: 56.7403
Epoch 1/10 | Step 50 | Loss: 45.9271
Epoch 1/10 | Step 60 | Loss: 44.5131
Epoch 1/10 | Step 70 | Loss: 63.5875
Epoch 1/10 | Step 80 | Loss: 58.9235
Epoch 1/10 | Step 90 | Loss: 59.8314
Epoch 1/10 | Step 100 | Loss: 90.9049
Epoch 1/10 | Step 110 | Loss: 63.9156
Epoch 1/10 | Step 120 | Loss: 54.8844
Epoch 1/10 | Step 130 | Loss: 42.1632
Epoch 1/10 | Step 140 | Loss: 44.8832
Epoch 1/10 | Step 150 | Loss: 55.8158
Epoch 1/10 | Step 160 | Loss: 60.9815
Epoch 1/10 | Step 170 | Loss: 64.4321
Epoch 1/10 | Step 180 | Loss: 57.8113
Epoch 1/10 | Step 190 | Loss: 46.3890
LEFT: torch.Size([1, 3, 288, 576])
RIGHT: torch.Size([1, 3, 288, 

In [28]:
torch.save(model.state_dict(), "mini_raft_kitti.pth")
